In [161]:
# Installation

!pip install -q crewai crewai-tools openai yfinance google-research-results litellm


ERROR: Could not find a version that satisfies the requirement google-research-results (from versions: none)
ERROR: No matching distribution found for google-research-results


In [162]:
pip install openai

In [163]:
!pip install -q \
crewai \
crewai-tools \
langchain \
langchain-community \
langchain-core \
langchain-google-genai \
yfinance \
google-search-results

In [164]:
import warnings
warnings.filterwarnings('ignore')
print('Installation complete!')

Installation complete!


In [165]:
# Import Liberaries
import os
from getpass import getpass
from crewai import Agent, Task, Crew, LLM
from crewai.tools import BaseTool
from typing import Type
from pydantic import BaseModel, Field
import yfinance as yf
from serpapi import GoogleSearch # use serpapi anytime you search google search (connecting to google search)
from crewai_tools import SerperDevTool


In [166]:
print('Libraries imported!')

Libraries imported!


In [167]:
from getpass import getpass
import os

print("Get OpenAI API KEY")
print("Get one free at: https://platform.openai.com/")
openai_api_key = getpass("Enter OpenAI API Key: ")

print("Get Serper API KEY")
print("Get one free at: https://serper.dev")
serper_api_key = getpass("Enter Serper API Key: ")

# ✅ Correct environment variables
os.environ["OPENAI_API_KEY"] = openai_api_key
os.environ["SERPER_API_KEY"] = serper_api_key

print("API keys configured!")

Get OpenAI API KEY
Get one free at: https://platform.openai.com/
Enter OpenAI API Key: ··········
Get Serper API KEY
Get one free at: https://serper.dev
Enter Serper API Key: ··········
API keys configured!


In [170]:
print(repr(openai_api_key[:12]))
print(repr(os.environ["OPENAI_API_KEY"][:12]))
print(repr(serper_api_key[:12]))
print(repr(os.environ["SERPER_API_KEY"][:12]))

'sk-proj-UXUn'
'sk-proj-UXUn'
'31c958df473a'
'31c958df473a'


In [171]:
#=====================================================================================
#Configure Openai LLM
#=====================================================================================
# Initialize Openai
llm = LLM(
    model="gpt-4o-mini",
    api_key=openai_api_key,
    temperature=0.3,
)

print("Openai LLM initialized ✅")

Openai LLM initialized ✅


In [172]:
from re import search
#Define Custom Tools

#Define input schemas
class StockSearchInput(BaseModel):
  query: str = Field(..., description="Search query for stock news")

class YahooFinanceInput(BaseTool):
  ticker: str = Field(..., description="Stock ticker symbol")


class StockSearchTool(BaseTool):
  name: str = "StockNewsSearcher"
  description: str = "Search for latest stock news using SerpAPI Google News"
  args_schema: Type[BaseModel] = StockSearchInput

  def _run(self, query: str) -> str:
    """Search for stock news using SerpAPI."""
    try:
        params = {
            "engine": "google",
            "q": query,
            "api_key": os.environ.get("SERPER_API_KEY"),
            "tbm": "nws",
            "hl": "en",
            "gl": "us",
            "num": 3,
        }

        search = GoogleSearch(params)
        results = search.get_dict()

        news = results.get("news_results", results.get("organic_results", []))

        if not news:
            return "No news found"

        output = []
        for item in news[:3]: #the first three news
            title = item.get("title", "No title")
            snippet = item.get("snippet", item.get("description", "No description"))
            link = item.get("link", "No link")

            output.append(
                f"Title: {title}\n"
                f"Snippet: {snippet}\n"
                f"Link: {link}\n"
            )

        return "\n".join(output)

    except Exception as e:
        return f"Error: {str(e)}"



In [173]:
from crewai.tools import BaseTool
import yfinance as yf

class YahooFinanceTool(BaseTool):
    name: str = "Yahoo Finance Tool"
    description: str = "Fetch recent stock price data from Yahoo Finance."
    #args_schema: Type[BaseModel] = YahooFinanceInput

    def _run(self, ticker: str) -> str:
        try:
            stock = yf.Ticker(ticker)
            hist = stock.history(period="1mo")

            if hist.empty:
                return f"No data found for {ticker}"

            latest = hist.tail(5)
            current = latest["Close"].iloc[-1]
            #start_price = latest["Close"].iloc[0]
            #change = latest["Close"].iloc[-1](current - start_price) / start_price * 100
            change = (latest["Close"].iloc[-1] - latest['Close'].iloc[0]) / latest['Close'].iloc[0]  * 100

            return (
                f"Stock: {ticker.upper()}\n"
                f"Price: ${current:.2f}\n"
                f"5-day Change: {change:+.2f}%\n"
                f"High: ${latest['High'].max():.2f}\n"
                f"Low: ${latest['Low'].min():.2f}"
            )
        except Exception as e:
            return f"Error: {str(e)}"

In [174]:
from crewai.tools import BaseTool

In [175]:
#initiate tools
search_tool=StockSearchTool()
yahoo_tool=YahooFinanceTool()
print("Tools initialized!")

Tools initialized!


In [176]:
print(type(search_tool))
print(type(yahoo_tool))

<class '__main__.StockSearchTool'>
<class '__main__.YahooFinanceTool'>


In [177]:
from crewai.tools import BaseTool

print(isinstance(search_tool, BaseTool))
print(isinstance(yahoo_tool, BaseTool))

True
True


In [178]:
# Define Agents
# Stock Analyst
from crewai import Agent
analyst=Agent(
    role="Stock Analyst",
    goal='Analyze stock data and news',
    backstory='Expert financial analyst',
    verbose=False,
    allow_delegation=False,
    llm=llm,
    tools=[search_tool,yahoo_tool]
)

In [179]:
#Report Writer
writer=Agent(
    role="Report Writer",
    goal='Write investment reports',
    backstory='professional financial report writer',
    verbose=False,
    allow_delegation=False,
    llm=llm,

)
print('Agents initialized')

Agents initialized


In [180]:
#===============================================================================
#Define Tasks
#===============================================================================
#Task 1: Search news
news_task=Task(
    description="""Search for the latest news about Apple Inc. (AAPL) stock".
    use query 'AAPL stock news'.
    Summerize the top 3 news items.""",
    expected_output='Summery of recent AAPL news',
    agent=analyst
)


In [181]:
#Task 2: Analayze stock price
price_task=Task(
    description="""Analayze Apple stock (AAPL) price trend".
    get price data and identify key trends.""",
    expected_output='price analysis for AAPL',
    agent=analyst
)


In [182]:
#Task 3: Writer report
report_task=Task(
    description="""Write a professional investor report for AAPL".
    Include:
    1. Executive Summery
    2. News Highlights
    3. Price Analysis
    4. Investment outlook

    Keep under 300 words.""",
    expected_output='Investment report for AAPL',
    agent=writer
)

print('Tasks defined!')

Tasks defined!


In [183]:
# Creat Crew
from crewai import Crew
#from tasks import NewsTask,PriceTask,ReportTask
crew=Crew(
    agents=[analyst,writer],
    tasks=[news_task,price_task,report_task],
    verbose=False,

)
print('\nStarting Stock Analysis....')
print('='*60)



Starting Stock Analysis....


In [184]:
from crewai import Crew

crew = Crew(
    agents=[analyst,writer],   # your agent(s)
    tasks=[news_task,price_task,report_task],       # your task(s)
    verbose=True
)

In [185]:
#Excute
result = await crew.kickoff_async()

print('\n' + '='*60)
print('Investment Report - Apple Inc. (AAPL)')
print('='*60)
print(result)
print('='*60)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 221f2258-0d82-47ee-b2e3-47c93cf56911                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search for the latest news about Apple Inc. (AAPL) stock".                                               │
│      use query 'AAPL stock news'.                                                                               │
│      Summerize the top 3 news items.                                                                            │
│  ID: 2c8830f5-dc09-44b9-8c46-053df4b1b324                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Analyst                                                                                           │
│                                                                                                                 │
│  Task: Search for the latest news about Apple Inc. (AAPL) stock".                                               │
│      use query 'AAPL stock news'.                                                                               │
│      Summerize the top 3 news items.                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: stock_news_searcher                                                                                      │
│  Args: {'query': 'AAPL stock news'}                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: stock_news_searcher                                                                                      │
│  Output: Title: Is Apple Stock (AAPL) a Buy Ahead of Q3 Earnings Report on July 30?                             │
│  Snippet: Apple ($AAPL) is scheduled to report its Q3 earnings on July 30, and investors are watching closely.  │
│  The company has faced slower iPhone...                                                                         │
│  Link: https://www.tipranks.com/news/is-apple-stock-aapl-a-buy-ahead-of-q3-earnings-report-on-july-30           │
│                                                                                                                 │
│  Title: Stock market trends: Apple soars, Tesla falters, healthcare shines                                      │
│  Snippet: Get the latest stock market news: Stock movers, top-performing sectors, and popular stock activity.   │
│  Stay ahead with a quick and...                                                                                 │
│  Link:                                                                                                          │
│  https://investinglive.com/stock-market-update/stock-market-trends-apple-soars-tesla-falters-healthcare-shines  │
│  -20260702/                                                                                                     │
│                                                                                                                 │
│  Title: 3 Reasons Apple Stock Is No Longer a Buy                                                                │
│  Snippet: When it comes to the top artificial intelligence (AI) stocks, it seems like investors pay less        │
│  attention to Apple (NASDAQ: AAPL) than in...                                                                   │
│  Link: https://finance.yahoo.com/markets/stocks/articles/3-reasons-apple-stock-no-142000617.html                │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Task: Write a professional investor report for AAPL".                                                          │
│      Include:                                                                                                   │
│      1. Executive Summery                                                                                       │
│      2. News Highlights                                                                                         │
│      3. Price Analysis                                                                                          │
│      4. Investment outlook                                                                                      │
│                                                                                                                 │
│      Keep under 300 words.                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Investment Report for Apple Inc. (AAPL)**                                                                    │
│                                                                                                                 │
│  **1. Executive Summary**                                                                                       │
│  Apple Inc. (AAPL) is currently trading at $308.63, reflecting an 8.76% increase over the past five days. As    │
│  the company prepares to release its Q3 earnings report on July 30, investor sentiment is mixed due to          │
│  concerns over slowing iPhone sales. Despite recent upward momentum, analysts caution that AAPL may not be the  │
│  best investment choice at this time.                                                                           │
│                                                                                                                 │
│  **2. News Highlights**                                                                                         │
│  - **Earnings Report Anticipation:** The upcoming Q3 earnings report is generating significant interest,        │
│  especially given the slowdown in iPhone sales which raises questions about future growth.                      │
│  - **Market Performance:** Apple has shown resilience in the stock market, outperforming competitors like       │
│  Tesla, which have faced recent challenges.                                                                     │
│  - **Investment Sentiment:** Analysts are divided, with some suggesting that AAPL may no longer be a favorable  │
│  investment due to a shift in focus towards other tech sectors, particularly artificial intelligence.           │
│                                                                                                                 │
│  **3. Price Analysis**                                                                                          │
│  - **Current Price:** $308.63                                                                                   │
│  - **5-day Change:** +8.76%                                                                                     │
│  - **High:** $309.42                                                                                            │
│  - **Low:** $274.21                                                                                             │
│                                                                                                                 │
│  AAPL has demonstrated a strong recovery from its recent low of $274.21. The stock is nearing a resistance      │
│  level at $309.42, which could indicate bullish sentiment if maintained. Investors should monitor this          │
│  critical price point closely.                                                                                  │
│                                                                                                                 │
│  **4. Investment Outlook**                                                                                      │
│  The outlook for AAPL remains cautiously optimistic. While the recent price trend is encouraging, the mixed     │
│  analyst sentiment and concerns over slowing sales may pose risks. Investors should consider the upcoming       │
│  earnings report as a pivotal moment for the stock. Key

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a professional investor report for AAPL".                                                          │
│      Include:                                                                                                   │
│      1. Executive Summery                                                                                       │
│      2. News Highlights                                                                                         │
│      3. Price Analysis                                                                                          │
│      4. Investment outlook                                                                                      │
│                                                                                                                 │
│      Keep under 300 words.                                                                                      │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Investment Report - Apple Inc. (AAPL)
**Investment Report for Apple Inc. (AAPL)**

**1. Executive Summary**  
Apple Inc. (AAPL) is currently trading at $308.63, reflecting an 8.76% increase over the past five days. As the company prepares to release its Q3 earnings report on July 30, investor sentiment is mixed due to concerns over slowing iPhone sales. Despite recent upward momentum, analysts caution that AAPL may not be the best investment choice at this time.

**2. News Highlights**  
- **Earnings Report Anticipation:** The upcoming Q3 earnings report is generating significant interest, especially given the slowdown in iPhone sales which raises questions about future growth.
- **Market Performance:** Apple has shown resilience in the stock market, outperforming competitors like Tesla, which have faced recent challenges.
- **Investment Sentiment:** Analysts are divided, with some suggesting that AAPL may no longer be a favorable investment due to a shift in focus towards other tech 

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 221f2258-0d82-47ee-b2e3-47c93cf56911                                                                       │
│  Final Output: **Investment Report for Apple Inc. (AAPL)**                                                      │
│                                                                                                                 │
│  **1. Executive Summary**                                                                                       │
│  Apple Inc. (AAPL) is currently trading at $308.63, reflecting an 8.76% increase over the past five days. As    │
│  the company prepares to release its Q3 earnings report on July 30, investor sentiment is mixed due to          │
│  concerns over slowing iPhone sales. Despite recent upward momentum, analysts caution that AAPL may not be the  │
│  best investment choice at this time.                                                                           │
│                                                                                                                 │
│  **2. News Highlights**                                                                                         │
│  - **Earnings Report Anticipation:** The upcoming Q3 earnings report is generating significant interest,        │
│  especially given the slowdown in iPhone sales which raises questions about future growth.                      │
│  - **Market Performance:** Apple has shown resilience in the stock market, outperforming competitors like       │
│  Tesla, which have faced recent challenges.                                                                     │
│  - **Investment Sentiment:** Analysts are divided, with some suggesting that AAPL may no longer be a favorable  │
│  investment due to a shift in focus towards other tech sectors, particularly artificial intelligence.           │
│                                                                                                                 │
│  **3. Price Analysis**                                                                                          │
│  - **Current Price:** $308.63                                                                                   │
│  - **5-day Change:** +8.76%                                                                                     │
│  - **High:** $309.42                                                                                            │
│  - **Low:** $274.21                                                                                             │
│                                                                                                                 │
│  AAPL has demonstrated a strong recovery from its recent low of $274.21. The stock is nearing a resistance      │
│  level at $309.42, which could indicate bullish sentiment if maintained. Investors should monitor this          │
│  critical price point closely.                                                                                  │
│                                                                                                                 │
│  **4. Investment Outlook**                                                                                      │
│  The outlook for AAPL remains cautiously optimistic. While the recent price trend is encouraging, the mixed     │
│  analyst sentiment and concerns over slowing sales may pose risks. Investors should consider the upcoming       │
│  earnings report as a pivotal moment for the stock. Ke